# use the data of a salmon farming site in the past few weeks to predict the salmon lice level for the next week.

In [ ]:
# Download data from API including week 1-52
from pathlib import Path
from getpass import getpass
import requests
import json
from datetime import date 
import time
raw_dir = Path("/Users/apple/Documents/GitHub/personlig-datavitenskapsportef-lje/ML-Salmon lice/raw/barentswatch_api")
raw_dir.mkdir(parents=True, exist_ok=True)
client_id = input("Enter client ID: ")
client_secret = getpass("Enter client secret: ")
def get_token():
    response=requests.post(
    "https://id.barentswatch.no/connect/token",
    data = {
    "grant_type": "client_credentials",
    "client_id": client_id,
    "client_secret": client_secret,
    "scope": "api"})
    response.raise_for_status()
    return response.json()['access_token']
token=get_token()
print('token obtained successfully') 


token obtained successfully


In [ ]:
def download_week(year, week, token):
    url = ("https://www.barentswatch.no/bwapi/"
           f"v2/geodata/fishhealth/locality/{year}/{week}")
    headers={"Authorization": f"Bearer {token}", 
             'Content-Type': 'application/json'}
    response=requests.post(url, headers=headers, 
                          json={},
                          timeout=60)
    return response

In [ ]:
def last_iso_week(year):
    return date(year, 12, 28).isocalendar().week
for year in range(2012, 2027):
    if year==2026: 
        final_week=33
    else:
        final_week=last_iso_week(year)
    print(f'\nDownloading{year}:week 1-{final_week}')
    for week in range(1, final_week+1):
        outfile=raw_dir / f'fishhealth_{year}_{week:02d}.json'
        if outfile.exists():
            print(f'{year} W{week:02d}: already downloaded')
            continue
        response=download_week(year, week, token)
        if response.status_code==401:
            print('toke expired, getting new token')
            token=get_token()
            response=download_week(year, week, token)
        if response.status_code==200:
            data=response.json()
            with open(outfile, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False)
            print(f'{year} W{week:02d}'
            f'{len(data)} localities')
        else:
            print (f'error{year} W({week:02d}:'
                   f'{response.status_code}')
            print(response.text[:500])

In [ ]:
import json 
import pandas as pd
from pathlib import Path
raw_dir=Path('/Users/apple/Documents/GitHub/personlig-datavitenskapsportef-lje/ML-Salmon lice/raw/barentswatch_api')
rows=[]
for file in raw_dir.glob('fishhealth_*.json'):
    parts=file.stem.split('_')
    year=int(parts[1])
    week=int(parts[2])
    with open(file, 'r', encoding='utf-8') as f:
        data=json.load(f)
    for x in data:
        locality=x.get('locality', {})
        municipality=x.get('municipality',{})
        geometry=x.get('geometry', {})
        lice=x.get('liceReport', {})
        coords=geometry.get('coordinates', [None, None])
        rows.append({'year': year,
                     'week': week,
                     'locality_week_id': x.get('localityWeekId'),
                     'locality_no': locality.get('no'),
                     'locality_name': locality.get('name'), 
                     'municipality_no': municipality.get('no'),
                     'municipality_name': municipality.get('name'), 
                     'longitude': coords[0] if len(coords) >0 else None,
                     'latitude': coords[1] if len(coords) > 1 else None, 
                     'has_salmonoid_license': x.get('hasSalmonoidLicense'),
                     'is_slaughter_holding_cage': x.get('isSlaughterHoldingCage'), 
                     'has_reported': lice.get('hasReported'),
                     'is_fallow': lice.get('isFallow'),
                     'adult_female_lice': lice.get('adultFemaleLice', {}).get('average'),
                     'mobile_lice': lice.get('mobileLice', {}).get('average'),
                     'stationary_lice': lice.get('stationaryLice', {}).get('average'),
                     'total_lice': lice.get('totalLice',{}).get('average'), 
                     'sea_temperature': lice.get('seaTemperature'), 
                     'n_treatments': len(x.get('liceTreatments', [])),
                     'n_diseases': len(x.get('diseases', []))}) 
df=pd.DataFrame(rows) #NB!: be careful about the indentation of this line, it should be outside the for loop, otherwise, the running time will be endless long
print(df.shape)
df.head()
df.info()
sorted(df['year'].unique())

In [ ]:
year_summary=df.groupby('year').agg(
    rows=('locality_no', 'size'),
    localities=('locality_no', 'nunique'), 
    weeks=('week', 'nunique'),
    reported=('has_reported', 'sum'),
    lice_nonmissing=('adult_female_lice', 'count'),
    temperature_nonmissing=('sea_temperature', 'count'))
year_summary['reported_pct']=(year_summary['reported']/year_summary['rows']*100)
year_summary['temperature_pct']=(year_summary['temperature_nonmissing']/year_summary['rows']*100)
year_summary.round(1)
#check report difference, we are interested in has_salmonoid_license==TRUE, is_fallow==FALSE, has_reported==TRUE
a=pd.crosstab(df['is_fallow'], df['has_reported'])
a

In [ ]:
model_base=df[(df['has_salmonoid_license']==True) & 
              (df['is_fallow']==False) &
              (df['has_reported']==True) 
              ]
print(model_base.shape)
model_base.groupby('year').agg(
    rows=('locality_no', 'size'),
    localities=('locality_no', 'nunique'),
    weeks=('week', 'nunique'))
model_base[
    [
        "adult_female_lice",
        "mobile_lice",
        "stationary_lice",
        "total_lice",
        "sea_temperature"
    ]
].describe() 
#here, the data distribution tend to be right skewed for all the lice variables, as the mean is larger than the median, and the max value is much larger than the 75% quantile.  
#also, the max temperature is 196.18? we mush check the data
model_base[
    [
        "adult_female_lice",
        "mobile_lice",
        "stationary_lice",
        "total_lice",
        "sea_temperature"
    ]
].quantile(
    [0.90, 0.95, 0.99, 0.995, 0.999, 1.0]
)

model_base[model_base['sea_temperature']>25].shape
model_base.loc[
    model_base["sea_temperature"] > 25,
    [
        "year",
        "week",
        "locality_no",
        "locality_name",
        "latitude",
        "longitude",
        "sea_temperature"]].sort_values("sea_temperature", ascending=False)

temp_check = model_base.sort_values(["locality_no", "year", "week"]).copy()
temp_check["temp_prev"] = (temp_check.groupby("locality_no")["sea_temperature"].shift(1))
temp_check["temp_next"] = (temp_check.groupby("locality_no")["sea_temperature"].shift(-1))
temp_outliers = temp_check[temp_check["sea_temperature"] > 25][
    [
        "year",
        "week",
        "locality_no",
        "locality_name",
        "sea_temperature",
        "temp_prev",
        "temp_next"]]
temp_outliers.head(30)

# as it is not decided to divide the temperature by 10 for all, and the outliers are not too much, make them missing value make more sense
import numpy as np
model_base['sea_temperature_clean']=model_base['sea_temperature'].copy() 
model_base.loc[model_base['sea_temperature_clean']>25, 'sea_temperature_clean'] = np.nan
model_base["sea_temperature_clean"].isna().sum()
model_base['calculated_sea_lice']=model_base['adult_female_lice'] + model_base['mobile_lice'] + model_base['stationary_lice'] 
model_base['total_dif']=model_base['calculated_sea_lice'] - model_base['total_lice']
model_base['total_dif'].sort_values()
model_base['total_dif'].sort_values()
model_base["total_dif"].describe(percentiles=[0.90, 0.99, 0.999])

In [ ]:
# start feature engineering, as the data are time series data, we can use the previous weeks' data to predict the next week's lice level. 
# but at the same time, we need to check the data consistency, as some localities may have missing weeks, if we want to use the the current week to predict the next week 
# the model logic: model_base -> active salmonoid + reported -> temperature_clean -> 7 days gap -> model ml
model_base['week_date']=pd.to_datetime(model_base['year'].astype(str) + '-W' + model_base['week'].astype(str).str.zfill(2) + '-1', format='%G-W%V-%u') 
model_base=model_base.sort_values(['locality_no', 'week_date'])
model_base['days_to_next_record']=(model_base.groupby('locality_no')['week_date'].shift(-1) -model_base['week_date']).dt.days
model_base["adult_female_lice_next"] = (model_base.groupby("locality_no")["adult_female_lice"].shift(-1))
model_ml=model_base[(model_base['days_to_next_record']==7) & (model_base['sea_temperature_clean'].notna())]
model_ml

In [221]:
# feature engineering: create lag features
features=[
    'adult_female_lice',
    'mobile_lice',
    'stationary_lice', 
    'sea_temperature_clean', 
    'n_treatments',
    'week']
X = model_ml[features]
y = model_ml["adult_female_lice_next"]
train_data=model_ml['year']<=2022
validation_data=model_ml['year'].isin([2023, 2024])
test_data=model_ml['year']==2025 
out_of_time_data=model_ml['year']==2026 
X_train, y_train=X.loc[train_data], y.loc[train_data]
X_validation, y_validation=X.loc[validation_data], y.loc[validation_data]
X_test, y_test=X.loc[test_data], y.loc[test_data]
X_out_of_time, y_out_of_time=X.loc[out_of_time_data], y.loc[out_of_time_data]

In [ ]:
# evaluation of baseline model, create the most simple prediction model for future comparison
# persistence baseline
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np 
baseline_val_pred=X_validation['adult_female_lice']
baseline_val_mae=mean_absolute_error(y_validation, baseline_val_pred) 
baseline_rmse=np.sqrt(mean_squared_error(y_validation, baseline_val_pred)) 
print('baseline mae:', baseline_val_mae) #loss function, average absolute error
print('baseline rmse:', baseline_rmse) # cost function, root mean squared error

baseline mae: 0.08458571215865546
baseline rmse: 0.24842448591249322


In [ ]:
# training a simple linear regression model: 
from sklearn.linear_model import LinearRegression
linear_model=LinearRegression()
linear_model.fit(X_train, y_train)
validation_pred_linear=linear_model.predict(X_validation)
linear_val_mae=mean_absolute_error(y_validation, validation_pred_linear)
linear_val_rmse=np.sqrt(mean_squared_error(y_validation, validation_pred_linear))
print('linear_val_mae:', linear_val_mae)
print('linear_val_rmse:', linear_val_rmse)
coef_table=pd.DataFrame({'feature': X_train.columns, 
                         'coefficient': linear_model.coef_})
print(coef_table)
print('intercept:', linear_model.intercept_)
# the results show that the linear regression model has a higher mae and lower rmse than the persistence baseline model, which means the linear regression model is not a good model for this problem.

linear_val_mae: 0.09311656864324751
linear_val_rmse: 0.2241515247964826
                 feature  coefficient
0      adult_female_lice     0.549319
1            mobile_lice     0.040867
2        stationary_lice     0.006233
3  sea_temperature_clean     0.004060
4           n_treatments    -0.024460
5                   week     0.000192
intercept: 0.010391531447830976


In [231]:
# create a nonlinear model, here we can try gradient boosting model and random forest model
from sklearn.ensemble import HistGradientBoostingRegressor 
gb_model=HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=31, random_state=42)
gb_model.fit(X_train, y_train) 
validation_pred_gb=gb_model.predict(X_validation)
gb_val_mae=mean_absolute_error(y_validation, validation_pred_gb)
gb_val_rmse=np.sqrt(mean_squared_error(y_validation, validation_pred_gb)) 
print('gb_val_mae:', gb_val_mae)
print('gb_val_rmse:', gb_val_rmse)

gb_val_mae: 0.08287168521197222
gb_val_rmse: 0.22157930027642653


In [181]:
model_base["days_to_next_record"].value_counts().sort_index().head(20)

days_to_next_record
7.0      380558
14.0      15771
21.0       1563
28.0        556
35.0        217
42.0        143
49.0         97
56.0         91
63.0         77
70.0         82
77.0        136
84.0        186
91.0        194
98.0        206
105.0       172
112.0       157
119.0       163
126.0       156
133.0       145
140.0       145
Name: count, dtype: int64

In [ ]:
### modeling population define
has_salmonoid_license==True
has_reported==True
is_fallow==False
#prediction target define

# create time series features
# create lag features
# create rolling features
# Treatment features
# EDA 
# build baseline 
# build model: linear regression, random forest, gradient boosting
# time series split 
# indicator evaluation: regression: MAE, RMSE, R**2
# check generalisation 
# feature importance/interpretation 
# final output 


(np.int64(2012), np.int64(2025))

2012 52
2013 52
2014 52
2015 53
2016 52
2017 52
2018 52
2019 52
2020 53
2021 52
2022 52
2023 52
2024 52
2025 52
2026 53


In [22]:
df52 = parse_week(data52, 2022, 52)

print(df52.shape)
df52.head()

(1719, 20)


,year,week,locality_week_id,locality_no,locality_name,municipality_no,municipality_name,longitude,latitude,has_salmonoid_license,is_slaughter_holding_cage,has_reported,is_fallow,adult_female_lice,mobile_lice,stationary_lice,total_lice,sea_temperature,n_treatments,n_diseases
0,2022,52,1513161,14746,Aarsand,1811,Bindal,12.156933,65.045867,False,False,False,True,NaN,NaN,NaN,NaN,NaN,0,0
1,2022,52,1513531,31937,Abelsnes,4207,Flekkefjord,6.656650,58.238767,False,False,False,True,NaN,NaN,NaN,NaN,NaN,0,0
2,2022,52,1513820,45119,Abelsnes II,4207,Flekkefjord,6.657250,58.240267,False,False,False,True,NaN,NaN,NaN,NaN,NaN,0,0
3,2022,52,1512548,10665,Adamselv,5438,Lebesby,26.691333,70.408000,True,False,False,True,NaN,NaN,NaN,NaN,NaN,0,0
4,2022,52,1513483,29196,Adjetjohka,5430,Guovdageaidnu - Kautokeino,22.918715,68.944137,False,False,False,True,NaN,NaN,NaN,NaN,NaN,0,0


In [25]:
treated[0]

{'localityWeekId': 1511727,
 'isFiltered': True,
 'locality': {'no': 13341, 'name': 'Bakjestranda', 'isOnLand': False},
 'geometry': {'type': 'Point', 'coordinates': [4.940267, 61.82025]},
 'municipality': {'no': '4648', 'name': 'Bremanger'},
 'hasSalmonoidLicense': True,
 'isSlaughterHoldingCage': False,
 'diseases': [],
 'liceReport': {'hasReported': True,
  'isFallow': False,
  'adultFemaleLice': {'average': 0.06,
   'averageOfPreviousWeek': 0.27,
   'trend': 'Decreasing'},
  'mobileLice': {'average': 0.33,
   'averageOfPreviousWeek': 0.22,
   'trend': 'Increasing'},
  'stationaryLice': {'average': 0.19,
   'averageOfPreviousWeek': 0.0,
   'trend': 'Increasing'},
  'totalLice': {'average': 0.58000004,
   'averageOfPreviousWeek': 0.49,
   'trend': 'Increasing'},
  'seaTemperature': 7.37},
 'liceTreatments': ['MEDIKAMENTELL']}